In [ ]:
from Bio import SeqIO
from collections import Counter
import pandas as pd 
import re
import requests
import json
import time
import glob
import pprint
import seaborn as sns
import pylab as plt
from matplotlib.ticker import MaxNLocator
from matplotlib.colors import ListedColormap
import pycountry 
import pycountry_convert as pc
pp = pprint.PrettyPrinter()
import numpy as np
from tabulate import tabulate
import geopandas as gpd
from geopandas import GeoDataFrame
import matplotlib.pyplot as plt
from shapely.geometry import Point, Polygon
import regex as re
import plotly.graph_objects as go
import os

from string import punctuation
from nltk.stem.snowball import SnowballStemmer
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk.corpus import wordnet
import nltk
from bs4 import BeautifulSoup
 

stemmer=SnowballStemmer("english")
import sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

import networkx as nx
from collections import defaultdict
from itertools import combinations
import sys
import ast

  


In [ ]:
# Resolve the repository and external data directory without machine-specific paths.
from pathlib import Path
import sys

for _candidate in (Path.cwd(), *Path.cwd().parents):
    if (_candidate / "project_config.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError("Run this notebook from within the cloned repository.")

from project_config import REPO_ROOT, DATA_ROOT, require_env

from pathlib import Path
import sys

# Code is run from the OneDrive project code folder.
CODE_ROOT = REPO_ROOT
DATASET_CURATION_CODE_ROOT = CODE_ROOT / "dataset_curation"
FUNCTIONS_DIR = DATASET_CURATION_CODE_ROOT / "functions"
if not FUNCTIONS_DIR.exists():
    raise FileNotFoundError(
        f"Functions folder not found: {FUNCTIONS_DIR}. "
        "Copy code/dataset_curation/functions to the OneDrive code root or adjust CODE_ROOT."
    )

if str(FUNCTIONS_DIR) not in sys.path:
    sys.path.insert(0, str(FUNCTIONS_DIR))

# Data is read from and written to the share-drive project folder.
SHARE_ROOT = DATA_ROOT
# DATA_ROOT is configured through project_config.

# Keep legacy string-style path variable because the downstream cells use
# expressions like path + r"processed_data\...".
path = str(DATA_ROOT) + "\\"

import backboning


In [ ]:
df_raw = pd.read_csv(path+r'processed_data\merged_all.csv', index_col = 0)

In [ ]:
df_raw

In [ ]:
Counter(df_raw['dates']).most_common()

In [ ]:
df_scholarly = pd.read_excel(path+r'processed_data\scholarly_final.xlsx', index_col = 0)

In [ ]:
# overview of keywords/subject fields

In [ ]:
fields_of_study_clean = []
for fields in df_scholarly['fields_of_study'].dropna().to_list():
    for field in ast.literal_eval(fields): 
        fields_of_study_clean.append(field)
    

In [ ]:
Counter(fields_of_study_clean).most_common()

In [ ]:
import networkx as nx
from itertools import combinations

# list of lists: each inner list = keywords for one paper
papers_keywords = df_scholarly['fields_of_study'].dropna().to_list()[:10000]

edge_w = Counter()
nodes = set()

for kws in papers_keywords:
    kws = list(dict.fromkeys(ast.literal_eval(kws)))  # de-duplicate within a paper
    nodes.update(kws)
    for a, b in combinations(kws, 2):
        if a != b:
            edge_w[tuple(sorted((a, b)))] += 1  # undirected weight = co-occurrence count

G = nx.Graph()
for n in nodes:
    G.add_node(n, label=n)

for (a, b), w in edge_w.items():
    G.add_edge(a, b, weight=w)

# Gephi-friendly graph file
nx.write_gexf(G, path+r"processed_data\keyword_cooccurrence.gexf")

In [ ]:

subjects_clean = []
for fields in df_scholarly['asjc_subjects'].dropna().to_list():
    for field in ast.literal_eval(fields): 
        subjects_clean.append(field)
    

In [ ]:
Counter(subjects_clean).most_common()

In [ ]:
#time

In [ ]:
Counter(df_scholarly['dates']).most_common()

In [ ]:
#researcher affiliation countries
df_scholarly

In [ ]:
lst = [x.strip() for x in fields.split(';')]
lst

In [ ]:

affiliations_countries_clean = []
for fields in df_scholarly['affiliations_countries'].dropna().to_list():
    lst = [x.strip() for x in fields.split(';')]

    for aff_cou in lst: 
        affiliations_countries_clean.append(aff_cou)
    

In [ ]:
Counter(affiliations_countries_clean).most_common()

## trash

In [ ]:
#df_scholarly_papers_sample = df_scholarly.sample(100000)

In [ ]:
df_id_title = [['lens_id', 'title']].drop_duplicates()
df_id_title.to_csv(path+R'processed_data\id_title_dict.csv')

In [ ]:
titles = list(set(df_scholarly.dropna(subset = ['title'])['title']))

In [ ]:
# Initialize lemmatizer
lemmatizer = WordNetLemmatizer()

# Function to map NLTK POS tags to WordNet POS tags
def get_wordnet_pos(nltk_pos_tag):
    if nltk_pos_tag.startswith('J'):
        return wordnet.ADJ
    elif nltk_pos_tag.startswith('V'):
        return wordnet.VERB
    elif nltk_pos_tag.startswith('N'):
        return wordnet.NOUN
    elif nltk_pos_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN  # Default to noun

# Function to lemmatize a title with POS tagging
def lemmatize_title(title):
    words = word_tokenize(title.lower())
    pos_tagged_words = nltk.pos_tag(words)
    
    lemmatized_words = [
        lemmatizer.lemmatize(word, get_wordnet_pos(pos_tag)) 
        for word, pos_tag in pos_tagged_words
    ]
    
    return ' '.join(lemmatized_words)

lemmatized_titles = []


for title in titles: 
    lemmatized_title = lemmatize_title(title)
    lemmatized_titles.append(lemmatized_title)

In [ ]:
# Define the set of spelled-out numbers to remove
spelled_out_numbers = {'zero', 'one', 'two', 'three', 'four', 'five', 'six', 'seven', 'eight', 'nine', 'ten',
                       'eleven', 'twelve', 'thirteen', 'fourteen', 'fifteen', 'sixteen', 'seventeen', 'eighteen',
                       'nineteen', 'twenty', 'thirty', 'forty', 'fifty', 'sixty', 'seventy', 'eighty', 'ninety', 
                       'hundred', 'thousand', 'million', 'billion','first', 'firstly', 'second', 'secondly', 'third', 'thirdly'}

# Define the set of countries, geographic regions, and their adjectives (demonyms) to remove
countries_and_regions = {'afghanistan', 'albania', 'algeria', 'andorra', 'angola', 'argentina', 'armenia', 'australia', 
                         'austria', 'azerbaijan', 'bahamas', 'bahrain', 'bangladesh', 'barbados', 'belarus', 'belgium', 
                         'belize', 'benin', 'bhutan', 'bolivia', 'bosnia', 'botswana', 'brazil', 'bulgaria', 'burkina', 
                         'burundi', 'cambodia', 'cameroon', 'canada', 'chad', 'chile', 'china', 'colombia', 'comoros', 
                         'congo', 'costa rica', 'croatia', 'cuba', 'cyprus', 'czech', 'denmark', 'djibouti', 'dominica', 'ecuador', 
                         'egypt', 'eritrea', 'estonia', 'eswatini', 'ethiopia', 'fiji', 'finland', 'france', 'gabon', 
                         'gambia', 'georgia', 'germany', 'ghana', 'greece', 'grenada', 'guatemala', 'guinea', 'guyana', 
                         'haiti', 'honduras', 'hungary', 'iceland', 'india', 'indonesia', 'iran', 'iraq', 'ireland', 
                         'israel', 'italy', 'jamaica', 'japan', 'jordan', 'kazakhstan', 'kenya', 'kiribati', 'kuwait', 
                         'kyrgyzstan', 'laos', 'latvia', 'lebanon', 'lesotho', 'liberia', 'libya', 'liechtenstein', 
                         'lithuania', 'luxembourg', 'madagascar', 'malawi', 'malaysia', 'maldives', 'mali', 'malta', 
                         'mauritania', 'mauritius', 'mexico', 'moldova', 'monaco', 'mongolia', 'montenegro', 'morocco', 
                         'mozambique', 'myanmar', 'namibia', 'nepal', 'netherlands', 'new zealand', 'nicaragua', 'niger', 
                         'nigeria', 'north korea', 'north macedonia', 'norway', 'oman', 'pakistan', 'palau', 'panama', 
                         'papua new guinea', 'paraguay', 'peru', 'philippines', 'poland', 'portugal', 'qatar', 'romania', 
                         'russia', 'rwanda', 'samoa', 'san marino', 'saudi arabia', 'senegal', 'serbia', 'seychelles', 
                         'sierra leone', 'singapore', 'slovakia', 'slovenia', 'somalia', 'south africa', 'south korea', 
                         'spain', 'sri lanka', 'sudan', 'suriname', 'sweden', 'switzerland', 'syria', 'tajikistan', 
                         'tanzania', 'thailand', 'togo', 'tonga', 'tunisia', 'turkey', 'turkmenistan', 'uganda', 'ukraine', 
                         'united arab emirates', 'united kingdom', 'united states', 'uruguay', 'uzbekistan', 'vanuatu', 
                         'venezuela', 'vietnam', 'yemen', 'zambia', 'zimbabwe',
                         # Geographic regions
                         'africa', 'asia', 'europe', 'america', 'north america', 'south america', 'oceania', 'antarctica', 
                         'middle east', 'eastern europe', 'western europe', 'northern europe', 'southern europe', 'latin america'}

# Add country-based adjectives (demonyms) to the list
demonyms = {'afghan', 'albanian', 'algerian', 'andorran', 'angolan', 'argentine', 'armenian', 'australian', 'austrian',
            'azerbaijani', 'bahamian', 'bahraini', 'bangladeshi', 'barbadian', 'belarusian', 'belgian', 'belizean', 
            'beninese', 'bhutanese', 'bolivian', 'bosnian', 'botswanan', 'brazilian', 'bulgarian', 'burkinabe', 
            'burundian', 'cambodian', 'cameroonian', 'canadian', 'chadian', 'chilean', 'chinese', 'colombian', 'comorian', 
            'congolese', 'costa rican','croatian', 'cuban', 'cypriot', 'czech', 'danish', 'djiboutian', 'dominican', 'ecuadorian', 
            'egyptian', 'eritrean', 'estonian', 'eswatini', 'ethiopian', 'fijian', 'finnish', 'french', 'gabonese', 
            'gambian', 'georgian', 'german', 'ghanaian', 'greek', 'grenadian', 'guatemalan', 'guinean', 'guyanese', 
            'haitian', 'honduran', 'hungarian', 'icelandic', 'indian', 'indonesian', 'iranian', 'iraqi', 'irish', 
            'israeli', 'italian', 'jamaican', 'japanese', 'jordanian', 'kazakh', 'kenyan', 'kiribati', 'kuwaiti', 
            'kyrgyz', 'laotian', 'latvian', 'lebanese', 'lesothan', 'liberian', 'libyan', 'liechtensteiner', 'lithuanian', 
            'luxembourger', 'malagasy', 'malawian', 'malaysian', 'maldivian', 'malian', 'maltese', 'mauritanian', 
            'mauritian', 'mexican', 'moldovan', 'monacan', 'mongolian', 'montenegrin', 'moroccan', 'mozambican', 'myanmarese', 
            'namibian', 'nepalese', 'dutch', 'new zealander', 'nicaraguan', 'nigerien', 'nigerian', 'north korean', 
            'macedonian', 'norwegian', 'omani', 'pakistani', 'palauan', 'panamanian', 'papua new guinean', 'paraguayan', 
            'peruvian', 'philippine', 'polish', 'portuguese', 'qatari', 'romanian', 'russian', 'rwandan', 'samoan', 
            'san marinese', 'saudi', 'senegalese', 'serbian', 'seychellois', 'sierra leonean', 'singaporean', 'slovak', 
            'slovenian', 'somali', 'south african', 'south korean', 'spanish', 'sri lankan', 'sudanese', 'surinamese', 
            'swazi', 'swedish', 'swiss', 'syrian', 'tajik', 'tanzanian', 'thai', 'togolese', 'tongan', 'tunisian', 
            'turkish', 'turkmen', 'ugandan', 'ukrainian', 'emirati', 'british', 'american', 'uruguayan', 'uzbek', 
            'vanuatuan', 'venezuelan', 'vietnamese', 'yemeni', 'zambian', 'zimbabwean', 'african', 'asian', 'european',  'mediterrenean', 'pacific', 'atlantic', 'usa', 'korea', 'republic', 'taiwan'}

# Merge countries, regions, and demonyms into a single set for filtering
countries_regions_demonyms = countries_and_regions.union(demonyms)

# Define additional stopwords if needed
own_stopwords = ['ci', 'sp', 'spp', 'title', 'summary', 'background','north', 'northern', 'south','southern','east','eastern','west','western', 'southwest', 'southeastern','southwest','southwestern', 'northeast','northeastern', 'northwest','northwestern',  'et', 'al', 'de', 'ii']
mystopwords = stopwords.words("english") + own_stopwords


titles_cleaned_stopwords = []


# Remove countries, regions, adjectives, numbers (written and actual), and other unwanted words
words_counter = Counter()
for title in lemmatized_titles:
    title_clean = ''
    for word in title.split():
        word_cleaned = "".join([l for l in word if l not in punctuation]).lower()
        
        # Remove stopwords, digits, spelled-out numbers, country names, regions, and adjectives (demonyms)
        if (word_cleaned not in mystopwords and 
            not word_cleaned.isdigit() and 
            word_cleaned not in spelled_out_numbers and 
            word_cleaned not in countries_regions_demonyms and 
            len(word_cleaned) > 1):
            #if word_cleaned in replace_dict.keys(): 
            #    word_cleaned = replace_dict[word_cleaned]
            
            if title_clean == '':
                title_clean = word_cleaned
            else:
                title_clean = title_clean + ' ' + word_cleaned
                
    titles_cleaned_stopwords.append(title_clean)
    words_counter.update(list(set(title_clean.split())))
titles_cleaned_stopwords[:10]


In [ ]:
words_counter.most_common(200)

In [ ]:
bottomfiltered_words_counter = Counter({key: value for key, value in words_counter.items() if value > 2})


In [ ]:
words_to_keep = list(bottomfiltered_words_counter.keys())

In [ ]:
titles_cleaned = []
titles_cleaned_stopwords_cut = []
#nr = 1
for title in titles_cleaned_stopwords:
    cleaned_title = ''
    for word in title.split(): 
        if word in words_to_keep:
            if cleaned_title == '': 
                cleaned_title = word
            else: 
                cleaned_title = cleaned_title + ' ' + word
    title_length = len(cleaned_title.split())

    titles_cleaned.append(cleaned_title)
    titles_cleaned_stopwords_cut.append(title)
    
    #if title_length <= 2: 
    #    print('smallllll')
    #    print(title)
    #    print(cleaned_title)
    #elif title_length >= 26: 
    #    print('largeee')
    #    print(title)
    #    print(cleaned_title)
    #nr = nr +1
    #print(nr)
    #if nr == 4: 
    #    break
    #if til

In [ ]:
ori_cle_title_dict = dict(zip(titles, titles_cleaned))

In [ ]:
len(titles_cleaned)

In [ ]:
df_scholarly['title_clean'] = df_scholarly['title'].map(ori_cle_title_dict)

In [ ]:
df_scholarly

In [ ]:
df_scholarly_clean = df_scholarly[df_scholarly['title_clean']!= '']

In [ ]:
#tf idf weighting
df_scholarly_clean_nd = df_scholarly_clean[['lens_id','title_clean']].drop_duplicates().dropna()

In [ ]:
titles_cleaned_nodup = df_scholarly_clean_nd['title_clean'].tolist()

In [ ]:
len(titles_cleaned_nodup)

In [ ]:
# Initialize the TfidfVectorizer
vectorizer = TfidfVectorizer() #max_df=0.95, min_df=4

# Fit the TF-IDF model and transform the titles into a sparse TF-IDF matrix
tfidf_matrix = vectorizer.fit_transform(titles_cleaned_nodup)

# Get the feature names (words) and their corresponding indices in the TF-IDF matrix
feature_names = vectorizer.get_feature_names_out()

# Step 2: Prepare lists to store log TF-IDF and log rank for averaging
max_rank = 1000  # Set a max rank cutoff to avoid extreme outliers; can be adjusted
log_tfidf_sums = np.zeros(max_rank)
rank_counts = np.zeros(max_rank)

# For each title (row in the tfidf_matrix), compute log TF-IDF vs log rank
for i in range(tfidf_matrix.shape[0]):
    # Get the non-zero indices and corresponding TF-IDF scores for the current title
    row = tfidf_matrix.getrow(i)
    non_zero_indices = row.indices
    non_zero_scores = row.data
    
    title_word_scores = pd.DataFrame({
        'word': feature_names[non_zero_indices],
        'tfidf_score': non_zero_scores
    })
    
    # Sort the words by TF-IDF score in descending order
    title_word_scores_sorted = title_word_scores.sort_values(by='tfidf_score', ascending=False)
    
    # Assign ranks
    title_word_scores_sorted['rank'] = np.arange(1, len(title_word_scores_sorted) + 1)
    
    # Remove any zero TF-IDF scores
    title_word_scores_sorted = title_word_scores_sorted[title_word_scores_sorted['tfidf_score'] > 0]
    
    # Compute the logarithms of TF-IDF scores and ranks
    log_tfidf = np.log(title_word_scores_sorted['tfidf_score'])
    log_rank = np.log(title_word_scores_sorted['rank'])
    
    # Limit to max_rank to avoid very high ranks (optional)
    for j in range(min(len(log_tfidf), max_rank)):
        log_tfidf_sums[j] += log_tfidf.iloc[j]
        rank_counts[j] += 1

# Step 3: Calculate the average log TF-IDF scores for each rank
average_log_tfidf = log_tfidf_sums / rank_counts

# Generate a log rank axis for plotting (from 1 to max_rank)
log_rank_axis = np.log(np.arange(1, max_rank + 1))

In [ ]:
threshold = -2.15

# Step 4: Plot the average log TF-IDF scores vs log rank
plt.figure(figsize=(10, 6))
plt.plot(log_rank_axis, average_log_tfidf, label='Average across Titles')

# Add a threshold line 
plt.axhline(y=threshold, color='red', linestyle='--', label='Threshold: y = '+str(threshold))

plt.title('Average Log of TF-IDF Scores vs Log of Ranks')
plt.xlabel('Log Transformed Word Rank')
plt.ylabel('Log Transformed TF-IDF Score')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig(path+R"plots/tf_idf_weighting.pdf")
plt.savefig(path+R"plots/tf_idf_weighting.png")
plt.show()

In [ ]:
words_below_threshold = []

# Iterate over each title's TF-IDF scores to find words with log(TF-IDF) < threshold
for i in range(tfidf_matrix.shape[0]):
    # Get the non-zero indices and corresponding TF-IDF scores for the current title
    row = tfidf_matrix.getrow(i)
    non_zero_indices = row.indices
    non_zero_scores = row.data
    
    title_word_scores = pd.DataFrame({
        'word': feature_names[non_zero_indices],
        'tfidf_score': non_zero_scores
    })
    
    # Compute the logarithm of the TF-IDF scores
    title_word_scores['log_tfidf'] = np.log(title_word_scores['tfidf_score'])
    
    # Filter words with log-transformed TF-IDF score below threshold
    filtered_words = title_word_scores[title_word_scores['log_tfidf'] < threshold]
    
    # Append the filtered words to the result list
    words_below_threshold.extend(filtered_words['word'].tolist())

# Step 2: Remove duplicates (since the same word might appear across different titles)
unique_words_below_threshold = list(set(words_below_threshold))

print("Words with log-transformed TF-IDF scores below "+str(threshold)+f": {len(unique_words_below_threshold)}")
print(unique_words_below_threshold)

pd.DataFrame(unique_words_below_threshold, columns=['word']).to_csv(path+R'processed_data\words_below_log_tfidf_'+str(threshold)+'.csv', index=False)

In [ ]:
#filter out the most generic words
titles_clean_tfidf = []
for title in titles_cleaned_nodup: 
    title_clean_tfidf = np.NaN
    for word in title.split(): 
        if word not in unique_words_below_threshold: 
            if pd.isna(title_clean_tfidf):
                title_clean_tfidf= word 
            else: 
                title_clean_tfidf = title_clean_tfidf + ' ' + word
    titles_clean_tfidf.append(title_clean_tfidf)
    
clean_tfidf_dict = dict(zip(titles_cleaned_nodup, titles_clean_tfidf))

In [ ]:
df_scholarly_clean['title_clean_tfidf'] = df_scholarly_clean['title_clean'].map(clean_tfidf_dict)

In [ ]:
df_scholarly_clean

In [ ]:
df_scholarly_clean[df_scholarly_clean['title_clean_tfidf'] != ''].to_csv(path+R'processed_data\scholarly_tfidf_cleaned_target_6.csv')

In [ ]:
#export the word - paper connections to re-connect after the community detection 
df_scholarly_clean_tfidf_nd= df_scholarly_clean[['lens_id','title_clean_tfidf']].drop_duplicates()

In [ ]:
from collections import defaultdict

# Create the word-to-ID dictionary
word_id_dict = defaultdict(set)  # Store sets of IDs for each word

# Iterate through the DataFrame to build the dictionary
for _, row in df_scholarly_clean_tfidf_nd[df_scholarly_clean_tfidf_nd['title_clean_tfidf'].notna()].iterrows():
    title = row['title_clean_tfidf']
    pub_id = row['lens_id']
    words = title.split()  # Split the title into words
    for word in words:
        word_id_dict[word].add(pub_id)

In [ ]:
# Convert sets to lists to make them JSON serializable
word_id_dict_serializable = {word: list(ids) for word, ids in word_id_dict.items()}

# Save to a JSON file
with open(path+R'processed_data\word_id_dict_tfidf.json', 'w') as json_file:
    json.dump(word_id_dict_serializable, json_file, indent=4)

## 2. Word-co occurance network

### 2.1. tf-idf network

In [ ]:
df_scholarly_clean_nd = df_scholarly_clean[['lens_id','title_clean_tfidf']].drop_duplicates().dropna()

In [ ]:
titles_cleaned_nd = df_scholarly_sample_clean_nd['title_clean_tfidf'].tolist()

In [ ]:
len(titles_cleaned_nd)

In [ ]:
#creat the word-combination edgesx
cooc=defaultdict(int)
for title in titles_cleaned_nd: 
    words=title.split()
    for a,b in set(combinations(words,2)): 
        if (b,a) in cooc: 
            a,b = b,a 
        if a!=b: 
            cooc[(a,b)]+=1 
            
#for combi in sorted(cooc,key=cooc.get,reverse=True): 
#    print (cooc[combi],"\t",combi)

In [ ]:
origin = []
destiny = []
count = []
for edge in list(cooc.keys()): 
    origin.append(edge[0])
    destiny.append(edge[1])
    count.append(cooc[edge])
    

In [ ]:
# Save data for backboning
df_coocurrance = pd.DataFrame(zip(origin, destiny, count), columns = ['source','target','weight'])
df_coocurrance.to_csv(path+R'processed_data\co_word_network_titles.csv')    


In [ ]:
# Save data for backboning
df_coocurrance_backboning = pd.DataFrame(zip(origin, destiny, count), columns = ['src','trg','nij'])

In [ ]:
df_coocurrance_backboning.to_csv(path+R'processed_data\co_word_network_titles_backboning.csv')    


In [ ]:
G = nx.from_pandas_edgelist(
    df_coocurrance,
    #edge_key="my_edge_key",
    edge_attr=["weight"],
    create_using=nx.Graph(),
    )

nx.write_gexf(G, path+R"processed_data\co_word_network.gexf")

In [ ]:
# Apply backboning process
nc_table = backboning.noise_corrected(df_coocurrance_backboning, undirected = True)
threshold_value = 0.9  # Adjust as required
nc_backbone_90 = backboning.thresholding(nc_table, threshold_value)

# Convert back to networkx graph if needed
G_backbone_90 = nx.from_pandas_edgelist(nc_backbone_90.rename(columns= {'nij':'weight'}), source='src', target='trg', edge_attr=True)

nx.write_gexf(G_backbone_90, path+R"processed_data\co_word_network_backbone_90.gexf")

nc_backbone_90

### 2.1. tf-idf network

In [ ]:
df_scholarly_clean_nd = df_scholarly_clean[['lens_id','title_clean']].drop_duplicates().dropna()
titles_cleaned_nd = df_scholarly_clean_nd['title_clean'].tolist()
len(titles_cleaned_nd)

In [ ]:
#creat the word-combination edgesx
cooc=defaultdict(int)
for title in titles_cleaned_nd: 
    words=title.split()
    for a,b in set(combinations(words,2)): 
        if (b,a) in cooc: 
            a,b = b,a 
        if a!=b: 
            cooc[(a,b)]+=1 
            
#for combi in sorted(cooc,key=cooc.get,reverse=True): 
#    print (cooc[combi],"\t",combi)

In [ ]:
origin = []
destiny = []
count = []
for edge in list(cooc.keys()): 
    origin.append(edge[0])
    destiny.append(edge[1])
    count.append(cooc[edge])

# Save data for backboning
df_coocurrance = pd.DataFrame(zip(origin, destiny, count), columns = ['source','target','weight'])
df_coocurrance.to_csv(path+R'processed_data\co_word_network_titles.csv')    

# Save data for backboning
df_coocurrance_backboning = pd.DataFrame(zip(origin, destiny, count), columns = ['src','trg','nij'])

df_coocurrance_backboning.to_csv(path+R'processed_data\co_word_network_titles_backboning.csv')    

G = nx.from_pandas_edgelist(
    df_coocurrance,
    #edge_key="my_edge_key",
    edge_attr=["weight"],
    create_using=nx.Graph(),
    )

nx.write_gexf(G, path+R"processed_data\co_word_network.gexf")

# Apply backboning process
nc_table = backboning.noise_corrected(df_coocurrance_backboning, undirected = True)
threshold_value = 0.9  # Adjust as required
nc_backbone_90 = backboning.thresholding(nc_table, threshold_value)

# Convert back to networkx graph if needed
G_backbone_90 = nx.from_pandas_edgelist(nc_backbone_90.rename(columns= {'nij':'weight'}), source='src', target='trg', edge_attr=True)

nx.write_gexf(G_backbone_90, path+R"processed_data\co_word_network_backbone_90.gexf")

nc_backbone_90